# Advanced Certification in AIML
## A Program by IIIT-H and TalentSprint

## Problem Statement

To identify and classify different types of weather using any open-source pre-trained models.

## Learning Objectives

At the end of this, you will be able to :

* Load and prepare the dataset
* Strengthen skills in the Transfer Learning area

In [0]:
#@title Mini-hackathon walkthrough
from IPython.display import HTML

HTML("""<video width="854" height="480" controls>
  <source src="https://cdn.iiith.talentsprint.com/aiml/Experiment_related_data/Walkthrough/B18_U4_MH_Walkthrough.mp4" type="video/mp4">
</video>
""")

## Dataset

Multi-class weather dataset(MWD) for image classification is a valuable dataset contains 1125 images divided into four folders such as sunrise, shine, rain, and cloudy.

## Grading = 10 Marks

## Setup Steps

In [0]:
#@title Run this cell to download the dataset

from IPython import get_ipython
ipython = get_ipython()

notebook="U4_MH1_Multi-class weather_Transfer_learning" #name of the notebook
Answer = "This notebook is graded by mentors on the day of hackathon"
def setup():
#
   ipython.magic("sx wget http://cdn.iiith.talentsprint.com/aiml/Experiment_related_data/Multi-classWeatherDataset.zip")
   ipython.magic("sx unzip -qq Multi-classWeatherDataset.zip")
   print ("Setup completed successfully")
   return

setup()

Setup completed successfully


## Basic Pytorch packages

**torchvision:**  This package is used to load and prepare the dataset. Using this package we can perform/apply transformations on the input data.

**transforms:**  This package is  used to perform preprocessing on images and operations sequentially.

**nn:**  This package provides an easy and modular way to build and train simple or complex neural networks.

**optim:** This package is used for  implementing various optimization algorithms

In [0]:
# Import Libraries
import torch
import torchvision
from torch import nn
from torch import optim
import torch.nn.functional as F

from torchsummary import summary
from torchvision import datasets, transforms, models

## **Stage 1 :** Data Preparation

2 Marks -> Load the data and Split the data into Train and Test sets

1. Define the transformations
2. Prepare the dataset with the defined transformations
3. Split the data into train and test sets
4. Load the train and test data using dataloader

In [0]:
import matplotlib.pyplot as plt
import numpy as np
import time

# Define device (GPU on Colab, CPU otherwise)
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f"Using device: {device}")

# 1. Define transforms (ImageNet normalization for pretrained models)
transform = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406],
                         std=[0.229, 0.224, 0.225])
])

# 2. Load full dataset using ImageFolder
dataset = datasets.ImageFolder(root='Multi-class Weather Dataset', transform=transform)
print(f"Classes: {dataset.classes}")
print(f"Class-to-index: {dataset.class_to_idx}")
print(f"Total images: {len(dataset)}")

# 3. Split into train (70%), validation (15%), test (15%)
train_size = int(0.70 * len(dataset))
val_size = int(0.15 * len(dataset))
test_size = len(dataset) - train_size - val_size
train_dataset, val_dataset, test_dataset = torch.utils.data.random_split(dataset, [train_size, val_size, test_size])
print(f"Train size: {len(train_dataset)}, Val size: {len(val_dataset)}, Test size: {len(test_dataset)}")

# 4. Create DataLoaders
train_loader = torch.utils.data.DataLoader(train_dataset, batch_size=32, shuffle=True)
val_loader = torch.utils.data.DataLoader(val_dataset, batch_size=32, shuffle=False)
test_loader = torch.utils.data.DataLoader(test_dataset, batch_size=32, shuffle=False)

1 Marks -> Plot the first 10 images

In [0]:
# Inverse normalization to display original images
inv_normalize = transforms.Normalize(
    mean=[-0.485/0.229, -0.456/0.224, -0.406/0.225],
    std=[1/0.229, 1/0.224, 1/0.225]
)

fig, axes = plt.subplots(2, 5, figsize=(15, 6))
class_names = dataset.classes

for i in range(10):
    img, label = dataset[i]
    img = inv_normalize(img)
    img = img.permute(1, 2, 0).numpy().clip(0, 1)
    ax = axes[i // 5][i % 5]
    ax.imshow(img)
    ax.set_title(class_names[label], fontsize=12)
    ax.axis('off')

plt.suptitle('First 10 Images from Multi-class Weather Dataset', fontsize=14)
plt.tight_layout()
plt.show()

## **Stage2:** Build the Model

2 Marks -> Prepare the Model using any open-source pretrained model

1. Download a pre-trained model.
2. Fine tune or customize layers in the output to perform classification on a specific task.
3. Declare the loss function and optimizer
4. Print the summary of the model.

In [0]:
# ==================== MODEL 1: ResNet18 ====================
resnet_model = models.resnet18(pretrained=True)

# Freeze all pretrained layers
for param in resnet_model.parameters():
    param.requires_grad = False

# Replace final FC layer for 4 weather classes (trainable)
num_features_resnet = resnet_model.fc.in_features
resnet_model.fc = nn.Linear(num_features_resnet, 4)
resnet_model = resnet_model.to(device)

# Loss and optimizer (only train the new FC layer)
resnet_criterion = nn.CrossEntropyLoss()
resnet_optimizer = optim.Adam(resnet_model.fc.parameters(), lr=0.001)

print("=" * 50)
print("MODEL 1: ResNet18 Summary")
print("=" * 50)
summary(resnet_model, (3, 224, 224))

# ==================== MODEL 2: VGG16 ====================
vgg_model = models.vgg16(pretrained=True)

# Freeze all pretrained layers
for param in vgg_model.parameters():
    param.requires_grad = False

# Replace final classifier layer for 4 weather classes (trainable)
num_features_vgg = vgg_model.classifier[6].in_features
vgg_model.classifier[6] = nn.Linear(num_features_vgg, 4)
vgg_model = vgg_model.to(device)

# Loss and optimizer (only train the new classifier layer)
vgg_criterion = nn.CrossEntropyLoss()
vgg_optimizer = optim.Adam(vgg_model.classifier[6].parameters(), lr=0.001)

print("\n" + "=" * 50)
print("MODEL 2: VGG16 Summary")
print("=" * 50)
summary(vgg_model, (3, 224, 224))

## **Stage 3:** Train the Model

2 Marks -> Train the Model and validate it continuously to calculate the loss and accuracy for the train dataset across each epoch.

**Expected training accuracy is above 95%**

In [0]:
def train_model(model, criterion, optimizer, train_loader, val_loader, num_epochs, model_name):
    """Train a model with validation tracking each epoch."""
    train_losses, val_losses = [], []
    train_accuracies, val_accuracies = [], []

    print(f"\n{'='*60}")
    print(f"Training {model_name}")
    print(f"{'='*60}")

    start_time = time.time()

    for epoch in range(num_epochs):
        # --- Training phase ---
        model.train()
        running_loss, correct, total = 0.0, 0, 0

        for images, labels in train_loader:
            images, labels = images.to(device), labels.to(device)
            optimizer.zero_grad()
            outputs = model(images)
            loss = criterion(outputs, labels)
            loss.backward()
            optimizer.step()

            running_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

        epoch_train_loss = running_loss / total
        epoch_train_acc = 100 * correct / total
        train_losses.append(epoch_train_loss)
        train_accuracies.append(epoch_train_acc)

        # --- Validation phase ---
        model.eval()
        val_running_loss, val_correct, val_total = 0.0, 0, 0

        with torch.no_grad():
            for images, labels in val_loader:
                images, labels = images.to(device), labels.to(device)
                outputs = model(images)
                loss = criterion(outputs, labels)

                val_running_loss += loss.item() * images.size(0)
                _, predicted = torch.max(outputs, 1)
                val_total += labels.size(0)
                val_correct += (predicted == labels).sum().item()

        epoch_val_loss = val_running_loss / val_total
        epoch_val_acc = 100 * val_correct / val_total
        val_losses.append(epoch_val_loss)
        val_accuracies.append(epoch_val_acc)

        print(f"Epoch [{epoch+1:2d}/{num_epochs}] | "
              f"Train Loss: {epoch_train_loss:.4f}, Train Acc: {epoch_train_acc:.2f}% | "
              f"Val Loss: {epoch_val_loss:.4f}, Val Acc: {epoch_val_acc:.2f}%")

    elapsed = time.time() - start_time
    print(f"\n{model_name} training completed in {elapsed:.1f}s")

    return train_losses, val_losses, train_accuracies, val_accuracies

# Record loss and accuracy of train + validation datasets
num_epochs = 15

# Train ResNet18
resnet_train_losses, resnet_val_losses, resnet_train_accs, resnet_val_accs = train_model(
    resnet_model, resnet_criterion, resnet_optimizer, train_loader, val_loader, num_epochs, "ResNet18"
)

# Train VGG16
vgg_train_losses, vgg_val_losses, vgg_train_accs, vgg_val_accs = train_model(
    vgg_model, vgg_criterion, vgg_optimizer, train_loader, val_loader, num_epochs, "VGG16"
)

# --- Plot: ResNet18 Train vs Validation ---
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
epochs_range = range(1, num_epochs + 1)

axes[0, 0].plot(epochs_range, resnet_train_losses, 'b-o', label='Train Loss')
axes[0, 0].plot(epochs_range, resnet_val_losses, 'r-s', label='Val Loss')
axes[0, 0].set_title('ResNet18 — Loss (Train vs Val)')
axes[0, 0].set_xlabel('Epoch'); axes[0, 0].set_ylabel('Loss')
axes[0, 0].legend(); axes[0, 0].grid(True)

axes[0, 1].plot(epochs_range, resnet_train_accs, 'b-o', label='Train Acc')
axes[0, 1].plot(epochs_range, resnet_val_accs, 'r-s', label='Val Acc')
axes[0, 1].set_title('ResNet18 — Accuracy (Train vs Val)')
axes[0, 1].set_xlabel('Epoch'); axes[0, 1].set_ylabel('Accuracy (%)')
axes[0, 1].legend(); axes[0, 1].grid(True)

# --- Plot: VGG16 Train vs Validation ---
axes[1, 0].plot(epochs_range, vgg_train_losses, 'b-o', label='Train Loss')
axes[1, 0].plot(epochs_range, vgg_val_losses, 'r-s', label='Val Loss')
axes[1, 0].set_title('VGG16 — Loss (Train vs Val)')
axes[1, 0].set_xlabel('Epoch'); axes[1, 0].set_ylabel('Loss')
axes[1, 0].legend(); axes[1, 0].grid(True)

axes[1, 1].plot(epochs_range, vgg_train_accs, 'b-o', label='Train Acc')
axes[1, 1].plot(epochs_range, vgg_val_accs, 'r-s', label='Val Acc')
axes[1, 1].set_title('VGG16 — Accuracy (Train vs Val)')
axes[1, 1].set_xlabel('Epoch'); axes[1, 1].set_ylabel('Accuracy (%)')
axes[1, 1].legend(); axes[1, 1].grid(True)

plt.suptitle('Overfitting Analysis: Train vs Validation Curves', fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# --- Overfitting gap summary ---
print(f"\n{'='*60}")
print(f"{'OVERFITTING GAP (Final Epoch)':^60}")
print(f"{'='*60}")
print(f"{'Model':<12} {'Train Acc':<12} {'Val Acc':<12} {'Gap':<12}")
print(f"{'-'*60}")
resnet_gap = resnet_train_accs[-1] - resnet_val_accs[-1]
vgg_gap = vgg_train_accs[-1] - vgg_val_accs[-1]
print(f"{'ResNet18':<12} {resnet_train_accs[-1]:<12.2f} {resnet_val_accs[-1]:<12.2f} {resnet_gap:<12.2f}")
print(f"{'VGG16':<12} {vgg_train_accs[-1]:<12.2f} {vgg_val_accs[-1]:<12.2f} {vgg_gap:<12.2f}")
print(f"{'='*60}")
if resnet_gap > 5 or vgg_gap > 5:
    print("⚠️  Gap > 5% detected — signs of overfitting. Consider data augmentation or dropout.")
else:
    print("✅ Both models generalize well — no significant overfitting.")

## **Stage 4:** Evaluate the Model

2 Marks -> Evaluate the model for the test dataset and calculate the loss and accuracy.

**Expected performance of test evaluation is above 95%**

In [0]:
def evaluate_model(model, criterion, data_loader, model_name, set_name="Test"):
    """Evaluate a model on any dataset split."""
    model.eval()
    total_loss = 0.0
    correct = 0
    total = 0

    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            loss = criterion(outputs, labels)

            total_loss += loss.item() * images.size(0)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()

    avg_loss = total_loss / total
    accuracy = 100 * correct / total

    print(f"{model_name} -- {set_name} Loss: {avg_loss:.4f}, {set_name} Accuracy: {accuracy:.2f}%")
    return avg_loss, accuracy

# --- Evaluate on HELD-OUT TEST SET (never seen during training) ---
print(f"\n{'='*60}")
print(f"{'FINAL TEST SET EVALUATION':^60}")
print(f"{'='*60}")
resnet_test_loss, resnet_test_acc = evaluate_model(
    resnet_model, resnet_criterion, test_loader, "ResNet18", "Test"
)
vgg_test_loss, vgg_test_acc = evaluate_model(
    vgg_model, vgg_criterion, test_loader, "VGG16", "Test"
)

# --- Full summary: Train vs Val vs Test ---
print(f"\n{'='*72}")
print(f"{'COMPLETE COMPARISON: Train / Validation / Test':^72}")
print(f"{'='*72}")
print(f"{'Model':<12} {'Train Acc':<12} {'Val Acc':<12} {'Test Acc':<12} {'Overfit Gap':<12}")
print(f"{'-'*72}")
resnet_gap = resnet_train_accs[-1] - resnet_val_accs[-1]
vgg_gap = vgg_train_accs[-1] - vgg_val_accs[-1]
print(f"{'ResNet18':<12} {resnet_train_accs[-1]:<12.2f} {resnet_val_accs[-1]:<12.2f} {resnet_test_acc:<12.2f} {resnet_gap:<12.2f}")
print(f"{'VGG16':<12} {vgg_train_accs[-1]:<12.2f} {vgg_val_accs[-1]:<12.2f} {vgg_test_acc:<12.2f} {vgg_gap:<12.2f}")
print(f"{'='*72}")

## **Stage 5:** Report your observations on any two open source pre-trained models

1 Marks -> Report your observations

In [0]:
"""
Observations: Comparing ResNet18 vs VGG16 on Multi-class Weather Dataset
(Based on actual training results on Google Colab T4 GPU)
================================================================================

1. ResNet18:
   - Architecture: 18 layers with residual (skip) connections
   - Parameters: ~11.7M total, only 2,052 trainable (FC layer: 512 -> 4)
   - Results:
     * Train Acc: 97.59% | Val Acc: 92.26% | Test Acc: 95.29%
     * Overfitting gap: 5.32% (train - val) -> WARNING sign
     * Val accuracy peaked at 95.83% (epoch 8) then degraded to 92.26%
     * Val loss started rising after epoch 14 (0.1821 -> 0.2023)
     * Training time: 113.6 seconds (fast)
   - Analysis:
     * Shows clear overfitting pattern: val accuracy drops while train rises
     * 512-dim FC bottleneck limits capacity -> memorizes instead of generalizing
     * Despite overfitting on validation, test accuracy (95.29%) met >95% target
     * Early stopping at epoch 8 would have yielded better val performance

2. VGG16:
   - Architecture: 16 layers with sequential conv blocks (no skip connections)
   - Parameters: ~138M total, only 16,388 trainable (classifier[6]: 4096 -> 4)
   - Results:
     * Train Acc: 98.60% | Val Acc: 97.02% | Test Acc: 94.71%
     * Overfitting gap: 1.58% (train - val) -> HEALTHY generalization
     * Val accuracy steadily improved, reaching 97.62% at epoch 13
     * Val loss remained stable and low throughout training
     * Training time: 173.0 seconds (1.5x slower than ResNet18)
   - Analysis:
     * Excellent generalization with minimal train-val gap
     * 4096-dim classifier features are richer for distinguishing 4 classes
     * Converged faster on validation (92.26% by epoch 1 vs ResNet18's 82.14%)
     * Test accuracy (94.71%) slightly below 95% target but very close

3. Key Findings:
   - VGG16 GENERALIZES BETTER (1.58% gap vs 5.32%) despite more parameters
   - ResNet18 slightly edges VGG16 on TEST accuracy (95.29% vs 94.71%)
   - VGG16's richer 4096-dim features help it learn more robust representations
   - ResNet18's overfitting can be mitigated with data augmentation or dropout

4. Conclusion:
   - Both models achieve ~95% test accuracy using transfer learning with frozen
     feature extractors and retrained classifier heads.
   - For GENERALIZATION: VGG16 is the better choice (tighter train-val gap)
   - For SPEED + DEPLOYMENT: ResNet18 wins (11.7M vs 138M params, 1.5x faster)
   - For PRODUCTION: ResNet18 with early stopping at epoch 8, or VGG16 as-is
   - Transfer learning is highly effective for small image datasets (1125 images)
     -- both ImageNet-pretrained models exceed 94% on weather classification.
"""

In [0]:
# ======================================================================
# Visualize intermediate layer activations for ResNet18 and VGG16
# This shows what features each layer extracts from a sample image
# ======================================================================

from torchvision.models.feature_extraction import create_feature_extractor

# --- Pick a sample image ---
sample_img, sample_label = dataset[0]
print(f"Sample image class: {class_names[sample_label]}")

# Show the original image
inv_img = inv_normalize(sample_img).permute(1, 2, 0).numpy().clip(0, 1)
fig, ax = plt.subplots(1, 1, figsize=(4, 4))
ax.imshow(inv_img)
ax.set_title(f'Input Image: {class_names[sample_label]}', fontsize=14)
ax.axis('off')
plt.show()

# ======================================================================
# Helper function to extract and plot feature maps from selected layers
# ======================================================================
def visualize_activations(model, input_tensor, layer_names, model_name, num_filters=8):
    """
    Extract activations from named layers and display the first `num_filters`
    feature maps for each layer.
    """
    extractor = create_feature_extractor(model, return_nodes=layer_names)
    model.eval()
    with torch.no_grad():
        activations = extractor(input_tensor.unsqueeze(0).to(device))

    num_layers = len(layer_names)
    fig, axes = plt.subplots(num_layers, num_filters, figsize=(num_filters * 2, num_layers * 2.5))
    if num_layers == 1:
        axes = [axes]  # make iterable

    for row, layer_name in enumerate(layer_names):
        act = activations[layer_name].squeeze(0).cpu()  # (C, H, W)
        for col in range(min(num_filters, act.shape[0])):
            axes[row][col].imshow(act[col], cmap='viridis')
            axes[row][col].axis('off')
            if col == 0:
                axes[row][col].set_ylabel(layer_name, fontsize=9, rotation=0, labelpad=80, va='center')
        # Turn off extra axes if fewer filters than num_filters
        for col in range(act.shape[0], num_filters):
            axes[row][col].axis('off')

    fig.suptitle(f'{model_name} — Feature Maps per Layer\n(first {num_filters} filters shown)',
                 fontsize=14, fontweight='bold')
    plt.tight_layout()
    plt.show()

    # Print activation shapes
    print(f"\n{model_name} activation shapes:")
    for name in layer_names:
        shape = activations[name].shape
        print(f"  {name:<30s} -> {list(shape)}")

# ======================================================================
# ResNet18: Visualize early, mid, and deep layers
# ======================================================================
resnet_layers = {
    'conv1':      'conv1',       # 7x7 conv, 64 filters (edges, textures)
    'layer1':     'layer1',      # First residual block (low-level patterns)
    'layer2':     'layer2',      # Mid-level features
    'layer3':     'layer3',      # High-level features
    'layer4':     'layer4',      # Final conv block (semantic features)
}
print("\n" + "="*60)
print("ResNet18 — Layer Activations")
print("="*60)
visualize_activations(resnet_model, sample_img, list(resnet_layers.values()), "ResNet18", num_filters=8)

# ======================================================================
# VGG16: Visualize activations from selected conv layers
# ======================================================================
vgg_layers = {
    'features.2':   'features.2',    # After 1st conv block (edges)
    'features.7':   'features.7',    # After 2nd conv block (textures)
    'features.14':  'features.14',   # After 3rd conv block (patterns)
    'features.24':  'features.24',   # After 4th conv block (parts)
    'features.30':  'features.30',   # Final conv layer (semantic)
}
print("\n" + "="*60)
print("VGG16 — Layer Activations")
print("="*60)
visualize_activations(vgg_model, sample_img, list(vgg_layers.values()), "VGG16", num_filters=8)

# ======================================================================
# Interpretation guide
# ======================================================================
print("""
======================================================================
INTERPRETATION GUIDE
======================================================================
Early layers (conv1, features.2):   Detect edges, colors, textures
Mid layers   (layer2, features.14): Detect patterns, shapes, parts
Deep layers  (layer4, features.30): Detect high-level semantic concepts
                                     (e.g., cloud formations, rain drops,
                                      sun glow, sky gradients)

Key observations:
- Early activations look similar across models (both learn edges)
- Deeper layers diverge: ResNet18 uses skip connections for sharper
  activations; VGG16 activations become more abstract and diffuse
- Brighter regions = stronger activation = model pays attention there
- If deep feature maps are mostly uniform → model may not be learning
  useful features (sign of underfitting or bad initialization)
======================================================================
""")

In [0]:
# ======================================================================
# Grad-CAM: Gradient-weighted Class Activation Mapping
# Shows which regions of the image each model focuses on for its prediction
# No external packages needed — implemented with PyTorch hooks
# ======================================================================
import torch.nn.functional as Fn

class GradCAM:
    """Grad-CAM implementation using forward/backward hooks."""
    def __init__(self, model, target_layer):
        self.model = model
        self.gradients = None
        self.activations = None

        # Register hooks on the target convolutional layer
        target_layer.register_forward_hook(self._forward_hook)
        target_layer.register_full_backward_hook(self._backward_hook)

    def _forward_hook(self, module, input, output):
        self.activations = output.detach()

    def _backward_hook(self, module, grad_input, grad_output):
        self.gradients = grad_output[0].detach()

    def generate(self, input_tensor, target_class=None):
        """Generate Grad-CAM heatmap for a given input image."""
        self.model.eval()
        input_tensor = input_tensor.unsqueeze(0).to(device).requires_grad_(True)

        # Forward pass
        output = self.model(input_tensor)
        if target_class is None:
            target_class = output.argmax(dim=1).item()

        # Backward pass from the target class score
        self.model.zero_grad()
        output[0, target_class].backward()

        # Compute Grad-CAM: global average pool gradients -> weight activations
        weights = self.gradients.mean(dim=(2, 3), keepdim=True)  # (1, C, 1, 1)
        cam = (weights * self.activations).sum(dim=1, keepdim=True)  # (1, 1, H, W)
        cam = Fn.relu(cam)  # Only keep positive contributions
        cam = cam.squeeze().cpu().numpy()

        # Normalize to [0, 1]
        cam = cam - cam.min()
        if cam.max() > 0:
            cam = cam / cam.max()

        return cam, target_class


def show_gradcam(model, target_layer, input_img, label, model_name, ax):
    """Generate and overlay Grad-CAM heatmap on the original image."""
    gradcam = GradCAM(model, target_layer)
    heatmap, pred_class = gradcam.generate(input_img)

    # Resize heatmap to match image size (224x224)
    heatmap_resized = np.array(
        transforms.functional.resize(
            torch.tensor(heatmap).unsqueeze(0), (224, 224)
        ).squeeze()
    )

    # Original image (inverse normalized)
    orig = inv_normalize(input_img).permute(1, 2, 0).numpy().clip(0, 1)

    # Overlay heatmap on image
    ax.imshow(orig)
    ax.imshow(heatmap_resized, cmap='jet', alpha=0.5)
    ax.set_title(f"{model_name}\nTrue: {class_names[label]} | Pred: {class_names[pred_class]}", fontsize=10)
    ax.axis('off')


# ======================================================================
# Grad-CAM target layers:
#   ResNet18 -> layer4[-1] (last residual block, 7x7 feature maps)
#   VGG16    -> features[30] (last conv layer before classifier)
# ======================================================================
resnet_target_layer = resnet_model.layer4[-1]
vgg_target_layer = vgg_model.features[30]

# --- Grad-CAM on multiple sample images (one per class if possible) ---
# Pick sample images: first occurrence of each class
sample_indices = []
seen_classes = set()
for i in range(len(dataset)):
    _, lbl = dataset[i]
    if lbl not in seen_classes:
        sample_indices.append(i)
        seen_classes.add(lbl)
    if len(seen_classes) == len(class_names):
        break

num_samples = len(sample_indices)
fig, axes = plt.subplots(3, num_samples, figsize=(num_samples * 4, 12))

for col, idx in enumerate(sample_indices):
    img, label = dataset[idx]

    # Row 0: Original image
    orig = inv_normalize(img).permute(1, 2, 0).numpy().clip(0, 1)
    axes[0, col].imshow(orig)
    axes[0, col].set_title(f"Original: {class_names[label]}", fontsize=11)
    axes[0, col].axis('off')

    # Row 1: ResNet18 Grad-CAM
    show_gradcam(resnet_model, resnet_target_layer, img, label, "ResNet18", axes[1, col])

    # Row 2: VGG16 Grad-CAM
    show_gradcam(vgg_model, vgg_target_layer, img, label, "VGG16", axes[2, col])

axes[0, 0].set_ylabel("Original", fontsize=12, fontweight='bold', rotation=0, labelpad=60, va='center')
axes[1, 0].set_ylabel("ResNet18\nGrad-CAM", fontsize=12, fontweight='bold', rotation=0, labelpad=70, va='center')
axes[2, 0].set_ylabel("VGG16\nGrad-CAM", fontsize=12, fontweight='bold', rotation=0, labelpad=70, va='center')

fig.suptitle('Grad-CAM: Where Each Model Looks (One Image Per Class)',
             fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# ======================================================================
# Interpretation
# ======================================================================
print("""
======================================================================
GRAD-CAM INTERPRETATION GUIDE
======================================================================
Red/Yellow regions = HIGH attention (model relies on these for prediction)
Blue/Green regions = LOW attention (ignored by the model)

What to look for:
- Cloudy:  Model should focus on cloud texture and grey sky regions
- Rain:    Model should focus on rain streaks, wet surfaces, dark skies
- Shine:   Model should focus on bright sky, sun glare, clear blue areas
- Sunrise: Model should focus on horizon line, warm color gradients

ResNet18 vs VGG16 differences:
- ResNet18 often produces more focused, localized attention maps
  (skip connections help preserve spatial precision)
- VGG16 tends to produce broader, more diffuse attention maps
  (sequential convolutions gradually lose spatial detail)

If both models attend to the same regions -> robust, reliable features
If attention differs -> models use different visual cues for same class
======================================================================
""")

In [0]:
# ======================================================================
# Confusion Matrix + Per-Class Precision, Recall, F1 for both models
# ======================================================================
from sklearn.metrics import confusion_matrix, classification_report
import itertools

def get_predictions(model, data_loader):
    """Collect all predictions and true labels from a data loader."""
    model.eval()
    all_preds = []
    all_labels = []
    with torch.no_grad():
        for images, labels in data_loader:
            images = images.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            all_preds.extend(predicted.cpu().numpy())
            all_labels.extend(labels.numpy())
    return np.array(all_labels), np.array(all_preds)


def plot_confusion_matrix(cm, classes, model_name, ax):
    """Plot a single confusion matrix with counts and percentages."""
    cm_pct = cm.astype('float') / cm.sum(axis=1, keepdims=True) * 100

    ax.imshow(cm, interpolation='nearest', cmap=plt.cm.Blues)
    ax.set_title(f'{model_name} — Confusion Matrix', fontsize=13, fontweight='bold')

    tick_marks = np.arange(len(classes))
    ax.set_xticks(tick_marks)
    ax.set_xticklabels(classes, rotation=45, ha='right', fontsize=10)
    ax.set_yticks(tick_marks)
    ax.set_yticklabels(classes, fontsize=10)

    # Annotate each cell with count and percentage
    thresh = cm.max() / 2.0
    for i, j in itertools.product(range(cm.shape[0]), range(cm.shape[1])):
        color = 'white' if cm[i, j] > thresh else 'black'
        ax.text(j, i, f'{cm[i, j]}\n({cm_pct[i, j]:.1f}%)',
                ha='center', va='center', color=color, fontsize=11)

    ax.set_ylabel('True Label', fontsize=11)
    ax.set_xlabel('Predicted Label', fontsize=11)


# --- Get predictions on TEST set ---
resnet_true, resnet_pred = get_predictions(resnet_model, test_loader)
vgg_true, vgg_pred = get_predictions(vgg_model, test_loader)

# --- Compute confusion matrices ---
resnet_cm = confusion_matrix(resnet_true, resnet_pred)
vgg_cm = confusion_matrix(vgg_true, vgg_pred)

# --- Plot side-by-side ---
fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 6))
plot_confusion_matrix(resnet_cm, class_names, 'ResNet18', ax1)
plot_confusion_matrix(vgg_cm, class_names, 'VGG16', ax2)
plt.suptitle('Test Set — Confusion Matrices', fontsize=15, fontweight='bold')
plt.tight_layout()
plt.show()

# --- Per-Class Classification Reports ---
print("\n" + "=" * 60)
print("ResNet18 — Per-Class Metrics (Test Set)")
print("=" * 60)
print(classification_report(resnet_true, resnet_pred, target_names=class_names, digits=3))

print("\n" + "=" * 60)
print("VGG16 — Per-Class Metrics (Test Set)")
print("=" * 60)
print(classification_report(vgg_true, vgg_pred, target_names=class_names, digits=3))

# --- Highlight which classes each model struggles with ---
print("\n" + "=" * 60)
print("PER-CLASS ACCURACY COMPARISON")
print("=" * 60)
print(f"{'Class':<12} {'ResNet18':<12} {'VGG16':<12} {'Better Model':<15}")
print(f"{'-'*60}")
for i, cls in enumerate(class_names):
    resnet_cls_acc = resnet_cm[i, i] / resnet_cm[i].sum() * 100 if resnet_cm[i].sum() > 0 else 0
    vgg_cls_acc = vgg_cm[i, i] / vgg_cm[i].sum() * 100 if vgg_cm[i].sum() > 0 else 0
    better = "ResNet18" if resnet_cls_acc >= vgg_cls_acc else "VGG16"
    print(f"{cls:<12} {resnet_cls_acc:<12.1f} {vgg_cls_acc:<12.1f} {better:<15}")
print(f"{'='*60}")

In [0]:
# ======================================================================
# Model Pruning: Global Unstructured Pruning on both models
# Compare model size, sparsity, and accuracy before vs after pruning
# ======================================================================
import torch.nn.utils.prune as prune
import copy
import os
import tempfile

# ======================================================================
# Helper functions
# ======================================================================
def count_parameters(model):
    """Count total and non-zero parameters."""
    total = 0
    nonzero = 0
    for param in model.parameters():
        total += param.numel()
        nonzero += (param != 0).sum().item()
    return total, nonzero

def get_model_size_mb(model):
    """Save model to a temp file and return size in MB."""
    with tempfile.NamedTemporaryFile(delete=True, suffix='.pt') as f:
        torch.save(model.state_dict(), f.name)
        size_mb = os.path.getsize(f.name) / (1024 * 1024)
    return size_mb

def get_sparsity(model):
    """Calculate overall sparsity (% of zero weights)."""
    total, nonzero = count_parameters(model)
    return 100 * (1 - nonzero / total)

def evaluate_accuracy(model, criterion, data_loader):
    """Quick accuracy evaluation."""
    model.eval()
    correct, total = 0, 0
    with torch.no_grad():
        for images, labels in data_loader:
            images, labels = images.to(device), labels.to(device)
            outputs = model(images)
            _, predicted = torch.max(outputs, 1)
            total += labels.size(0)
            correct += (predicted == labels).sum().item()
    return 100 * correct / total

def apply_global_pruning(model, amount):
    """Apply global unstructured L1 pruning to all Conv2d and Linear layers."""
    parameters_to_prune = []
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            parameters_to_prune.append((module, 'weight'))

    prune.global_unstructured(
        parameters_to_prune,
        pruning_method=prune.L1Unstructured,
        amount=amount,
    )
    return model

def remove_pruning_reparametrization(model):
    """Make pruning permanent by removing the forward hooks."""
    for name, module in model.named_modules():
        if isinstance(module, (nn.Conv2d, nn.Linear)):
            try:
                prune.remove(module, 'weight')
            except ValueError:
                pass
    return model

# ======================================================================
# Step 1: Record BEFORE-pruning metrics
# ======================================================================
print("=" * 70)
print("BEFORE PRUNING")
print("=" * 70)

# Deep copy models so originals are preserved
resnet_pruned = copy.deepcopy(resnet_model)
vgg_pruned = copy.deepcopy(vgg_model)

resnet_total_before, resnet_nz_before = count_parameters(resnet_model)
vgg_total_before, vgg_nz_before = count_parameters(vgg_model)
resnet_size_before = get_model_size_mb(resnet_model)
vgg_size_before = get_model_size_mb(vgg_model)
resnet_acc_before = evaluate_accuracy(resnet_model, resnet_criterion, test_loader)
vgg_acc_before = evaluate_accuracy(vgg_model, vgg_criterion, test_loader)
resnet_sparsity_before = get_sparsity(resnet_model)
vgg_sparsity_before = get_sparsity(vgg_model)

print(f"{'Metric':<25} {'ResNet18':<20} {'VGG16':<20}")
print(f"{'-'*65}")
print(f"{'Total Parameters':<25} {resnet_total_before:<20,} {vgg_total_before:<20,}")
print(f"{'Non-zero Parameters':<25} {resnet_nz_before:<20,} {vgg_nz_before:<20,}")
print(f"{'Sparsity (%)':<25} {resnet_sparsity_before:<20.2f} {vgg_sparsity_before:<20.2f}")
print(f"{'Model Size (MB)':<25} {resnet_size_before:<20.2f} {vgg_size_before:<20.2f}")
print(f"{'Test Accuracy (%)':<25} {resnet_acc_before:<20.2f} {vgg_acc_before:<20.2f}")

# ======================================================================
# Step 2: Apply pruning at multiple sparsity levels
# ======================================================================
pruning_levels = [0.3, 0.5, 0.7]  # 30%, 50%, 70% pruning

results = {'ResNet18': [], 'VGG16': []}

for amount in pruning_levels:
    print(f"\n{'='*70}")
    print(f"PRUNING AT {int(amount*100)}% SPARSITY")
    print(f"{'='*70}")

    # --- ResNet18 ---
    resnet_p = copy.deepcopy(resnet_model)
    resnet_p = apply_global_pruning(resnet_p, amount)
    resnet_p_acc = evaluate_accuracy(resnet_p, resnet_criterion, test_loader)
    resnet_p = remove_pruning_reparametrization(resnet_p)
    resnet_p_total, resnet_p_nz = count_parameters(resnet_p)
    resnet_p_size = get_model_size_mb(resnet_p)
    resnet_p_sparsity = get_sparsity(resnet_p)

    results['ResNet18'].append({
        'amount': amount, 'acc': resnet_p_acc, 'size': resnet_p_size,
        'sparsity': resnet_p_sparsity, 'nonzero': resnet_p_nz
    })

    # --- VGG16 ---
    vgg_p = copy.deepcopy(vgg_model)
    vgg_p = apply_global_pruning(vgg_p, amount)
    vgg_p_acc = evaluate_accuracy(vgg_p, vgg_criterion, test_loader)
    vgg_p = remove_pruning_reparametrization(vgg_p)
    vgg_p_total, vgg_p_nz = count_parameters(vgg_p)
    vgg_p_size = get_model_size_mb(vgg_p)
    vgg_p_sparsity = get_sparsity(vgg_p)

    results['VGG16'].append({
        'amount': amount, 'acc': vgg_p_acc, 'size': vgg_p_size,
        'sparsity': vgg_p_sparsity, 'nonzero': vgg_p_nz
    })

    print(f"{'Metric':<25} {'ResNet18':<20} {'VGG16':<20}")
    print(f"{'-'*65}")
    print(f"{'Actual Sparsity (%)':<25} {resnet_p_sparsity:<20.2f} {vgg_p_sparsity:<20.2f}")
    print(f"{'Non-zero Parameters':<25} {resnet_p_nz:<20,} {vgg_p_nz:<20,}")
    print(f"{'Model Size (MB)':<25} {resnet_p_size:<20.2f} {vgg_p_size:<20.2f}")
    print(f"{'Test Accuracy (%)':<25} {resnet_p_acc:<20.2f} {vgg_p_acc:<20.2f}")
    print(f"{'Accuracy Drop (%)':<25} {resnet_acc_before - resnet_p_acc:<20.2f} {vgg_acc_before - vgg_p_acc:<20.2f}")

# ======================================================================
# Step 3: Plot comparison charts
# ======================================================================
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

prune_labels = ['Original'] + [f'{int(a*100)}%' for a in pruning_levels]

# --- Accuracy vs Pruning Level ---
resnet_accs_list = [resnet_acc_before] + [r['acc'] for r in results['ResNet18']]
vgg_accs_list = [vgg_acc_before] + [r['acc'] for r in results['VGG16']]

x = np.arange(len(prune_labels))
width = 0.35
axes[0].bar(x - width/2, resnet_accs_list, width, label='ResNet18', color='steelblue')
axes[0].bar(x + width/2, vgg_accs_list, width, label='VGG16', color='coral')
axes[0].set_ylabel('Test Accuracy (%)')
axes[0].set_title('Accuracy vs Pruning Level')
axes[0].set_xticks(x)
axes[0].set_xticklabels(prune_labels)
axes[0].legend()
axes[0].grid(axis='y', alpha=0.3)
axes[0].set_ylim(max(0, min(resnet_accs_list + vgg_accs_list) - 10), 105)

# --- Model Size vs Pruning Level ---
resnet_sizes_list = [resnet_size_before] + [r['size'] for r in results['ResNet18']]
vgg_sizes_list = [vgg_size_before] + [r['size'] for r in results['VGG16']]

axes[1].bar(x - width/2, resnet_sizes_list, width, label='ResNet18', color='steelblue')
axes[1].bar(x + width/2, vgg_sizes_list, width, label='VGG16', color='coral')
axes[1].set_ylabel('Model Size (MB)')
axes[1].set_title('Model Size vs Pruning Level')
axes[1].set_xticks(x)
axes[1].set_xticklabels(prune_labels)
axes[1].legend()
axes[1].grid(axis='y', alpha=0.3)

# --- Non-zero Parameters vs Pruning Level ---
resnet_nz_list = [resnet_nz_before] + [r['nonzero'] for r in results['ResNet18']]
vgg_nz_list = [vgg_nz_before] + [r['nonzero'] for r in results['VGG16']]

axes[2].bar(x - width/2, [n/1e6 for n in resnet_nz_list], width, label='ResNet18', color='steelblue')
axes[2].bar(x + width/2, [n/1e6 for n in vgg_nz_list], width, label='VGG16', color='coral')
axes[2].set_ylabel('Non-zero Parameters (Millions)')
axes[2].set_title('Active Parameters vs Pruning Level')
axes[2].set_xticks(x)
axes[2].set_xticklabels(prune_labels)
axes[2].legend()
axes[2].grid(axis='y', alpha=0.3)

plt.suptitle('Model Pruning Analysis: Size vs Accuracy Trade-off',
             fontsize=14, fontweight='bold')
plt.tight_layout()
plt.show()

# ======================================================================
# Step 4: Summary and recommendation
# ======================================================================
print(f"\n{'='*70}")
print(f"{'PRUNING SUMMARY TABLE':^70}")
print(f"{'='*70}")
print(f"{'Model':<10} {'Pruning':<10} {'Accuracy':<12} {'Acc Drop':<12} {'Size (MB)':<12} {'Sparsity':<10}")
print(f"{'-'*70}")
print(f"{'ResNet18':<10} {'0%':<10} {resnet_acc_before:<12.2f} {'--':<12} {resnet_size_before:<12.2f} {resnet_sparsity_before:<10.1f}")
for r in results['ResNet18']:
    drop = resnet_acc_before - r['acc']
    print(f"{'ResNet18':<10} {int(r['amount']*100):>3}%{'':5} {r['acc']:<12.2f} {drop:<12.2f} {r['size']:<12.2f} {r['sparsity']:<10.1f}")
print(f"{'-'*70}")
print(f"{'VGG16':<10} {'0%':<10} {vgg_acc_before:<12.2f} {'--':<12} {vgg_size_before:<12.2f} {vgg_sparsity_before:<10.1f}")
for r in results['VGG16']:
    drop = vgg_acc_before - r['acc']
    print(f"{'VGG16':<10} {int(r['amount']*100):>3}%{'':5} {r['acc']:<12.2f} {drop:<12.2f} {r['size']:<12.2f} {r['sparsity']:<10.1f}")
print(f"{'='*70}")

# Find best trade-off (highest pruning with <2% accuracy drop)
print("\nRECOMMENDATION:")
for model_name, before_acc in [('ResNet18', resnet_acc_before), ('VGG16', vgg_acc_before)]:
    best = None
    for r in reversed(results[model_name]):
        if before_acc - r['acc'] < 2.0:
            best = r
            break
    if best:
        print(f"  {model_name}: Use {int(best['amount']*100)}% pruning "
              f"(accuracy drop: {before_acc - best['acc']:.2f}%, "
              f"sparsity: {best['sparsity']:.1f}%)")
    else:
        print(f"  {model_name}: All pruning levels cause >2% accuracy drop. "
              f"Fine-tune after pruning for better results.")

print("\nNOTE: The .pt file size stays similar because zeros are still stored.")
print("For real compression, export with torch.sparse or use ONNX + quantization.")